# Lab 09 — Relational Queries — Chinook Music DB
**SQL for Analysts Track** · Beginner–Intermediate · ~50 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Connect to SQLite and list schema tables
2. Write JOIN + GROUP BY revenue queries
3. Use LEFT JOIN anti-patterns to find orphan rows
4. Load SQL results into pandas and plot monthly revenue

## Datasets (this folder)
- `chinook.sqlite`

## How to run on Google Colab
1. Open this notebook from the Kongu AI Academy lab page (**Open in Colab**). Colab saves a copy to your Google Drive.
2. Run **Cell 0** first: it downloads and unzips the dataset.
3. Write your code in the exercise cells and run them.
4. Run the last cell, `kaa_check()`, and compare your answers with the expected outputs in the exercises.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`chinook.sqlite`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — Kongu AI Academy: download the dataset (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-09-chinook-sqlite"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/abhisek8113/kongu-ai-academy/main/public/course-labs/course-01-foundations-python-math-data/lab-09-chinook-sqlite/bundle/dataset.zip"
NEED = ["chinook.sqlite"]  # unzipped files used by the code below
EXERCISES = ["avg_invoice", "rock_revenue", "rock_share", "mean_spend"]  # functions you write in the exercise cells


def kaa_check():
    """Self-check: runs each exercise function and prints what it returns."""
    for name in EXERCISES:
        fn = globals().get(name)
        if not callable(fn):
            print(f"✗ {name}: not written yet")
            continue
        try:
            print(f"✓ {name}() ->", fn())
        except TypeError:
            print(f"✓ {name} is defined (takes arguments; test it with the example in its exercise)")
        except Exception as exc:
            print(f"✗ {name} raised {type(exc).__name__}: {exc}")


def _have_files():
    return all(os.path.exists(f) for f in NEED)


if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", DATASET_ZIP_URL, "-O", "dataset.zip"], check=True)
    else:
        urllib.request.urlretrieve(DATASET_ZIP_URL, "dataset.zip")
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))

## SQL for Analysts Track: JOIN, GROUP BY, Results → pandas

> **Scenario:** `chinook.sqlite` is a music-store schema (artists, albums, tracks, invoices, customers). Answer revenue and customer questions with SQL, then load results into pandas for a monthly trend.
>
> **You will learn:** `sqlite3.connect`, SELECT / JOIN / GROUP BY, subqueries, `pandas.read_sql`.
> **Time:** ~50 minutes. **Level:** Beginner–Intermediate. **Needs:** sqlite3 (stdlib) + pandas. **Env:** 🟢 Colab only.

### SQL mental map

| Spreadsheet idea | SQL | Python |
|---|---|---|
| Filter rows | `WHERE` | `df[df.col == …]` |
| VLOOKUP across sheets | `JOIN` | `merge` |
| PivotTable values | `SUM(...) GROUP BY` | `groupby().sum()` |
| Top N | `ORDER BY … LIMIT n` | `nlargest` |
| Two-step pivot | subquery / CTE | chain groupbys |

The middle column is the SQL you will type; the right column is the pandas equivalent if you ever forget what a clause means. Translate freely in both directions.

---

### 1. Connect and list tables (local first, Colab fallback)

SQLite needs no server: the whole catalog lives in one file and `sqlite3` is standard library. Start by reading `sqlite_master` — the database’s self-description — and capture two sanity totals (customer count, lifetime revenue). Later queries should reconcile to **59 customers** and ≈ **2328.60** in invoice totals; if they do not, the join path is wrong.


In [ ]:
import os, sqlite3
import pandas as pd

def connect_chinook():
    local = "chinook.sqlite"
    if not os.path.exists(local):
        import urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/lerocha/chinook-database/master/"
            "ChinookDatabase/DataSources/Chinook_Sqlite.sqlite",
            local,
        )
    return sqlite3.connect(local)

conn = connect_chinook()
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)["name"].tolist()
print(tables)
# ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice',
#  'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']

print(pd.read_sql("SELECT COUNT(*) AS n_customers FROM Customer", conn))
print(pd.read_sql("SELECT ROUND(SUM(Total),2) AS revenue FROM Invoice", conn))


Track revenue lives at the line-item grain, so start from `InvoiceLine` and join “upward” for display names only — each step is many-to-one, so no row multiplication occurs:


```text
Invoice ──< InvoiceLine >── Track ──< Album >── Artist
```


**What to notice:**
- **11 tables**; the revenue walk is `InvoiceLine → Track → Album → Artist`, with `Invoice` hanging off the same line items for document-level totals.
- Sanity totals: **59 customers**, revenue **2328.60** — reuse them as denominators (Rock share, mean month) later.
- Because every join points toward the “one” side of a relationship, `SUM` at the track grain in Section 2 is not inflated by the joins.

---

### 2. Top 5 tracks by revenue

Money is recorded per line item: `Quantity × UnitPrice` on each `InvoiceLine` row. Aggregate that at the track grain (`GROUP BY t.TrackId`), pull display names through INNER JOINs, then `ORDER BY revenue DESC LIMIT 5` — SQL’s version of a sorted PivotTable top-N.


In [ ]:
top5 = pd.read_sql("""
SELECT t.Name        AS track,
       al.Title      AS album,
       ar.Name       AS artist,
       ROUND(SUM(il.Quantity * il.UnitPrice), 2) AS revenue
FROM InvoiceLine il
JOIN Track t    ON t.TrackId  = il.TrackId
JOIN Album al   ON al.AlbumId = t.AlbumId
JOIN Artist ar  ON ar.ArtistId= al.ArtistId
GROUP BY t.TrackId
ORDER BY revenue DESC
LIMIT 5
""", conn)
print(top5.to_string(index=False))


Expected (ties at the top are common — unit prices cluster at 0.99×qty):

| track | album | artist | revenue |
|---|---|---|---|
| The Woman King | Battlestar Galactica, Season 3 | Battlestar Galactica | 3.98 |
| The Fix | Heroes, Season 1 | Heroes | 3.98 |
| Walkabout | Lost, Season 1 | Lost | 3.98 |
| Hot Girl | The Office, Season 1 | The Office | 3.98 |
| Gay Witch Hunt | The Office, Season 3 | The Office | 3.98 |

**What to notice:**
- All five revenues are **3.98** — ties are expected when unit prices cluster at 0.99×quantity, not a sign the query is broken.
- `GROUP BY t.TrackId` (the key, not the name) keeps identically titled tracks on different albums apart.
- SQL guarantees the revenue order but not which of the tied rows come first — ties, not errors.

> **Pitfall:** Joining before you aggregate needs care — joining two *one-to-many* branches (say, invoices and their line items) fans each line out across invoice columns and silently multiplies sums. Fix the grain first: aggregate from the line-item table, or join only toward many-to-one “lookup” tables for names, as this query does.


In [ ]:
top5.to_csv("top_tracks.csv", index=False)
print("wrote top_tracks.csv")


---

### 3. Customers with no purchases (LEFT JOIN anti-pattern)

An INNER JOIN would make never-customers disappear silently — unmatched rows are simply dropped. `LEFT JOIN` keeps every customer and writes NULLs into the invoice columns when no invoice matches; `WHERE i.InvoiceId IS NULL` then isolates exactly those orphan rows. This anti-join works on any schema: swap the tables, keep the pattern.


In [ ]:
no_buy = pd.read_sql("""
SELECT c.CustomerId, c.FirstName, c.LastName, c.Email
FROM Customer c
LEFT JOIN Invoice i ON i.CustomerId = c.CustomerId
WHERE i.InvoiceId IS NULL
ORDER BY c.CustomerId
""", conn)
print(len(no_buy), "customers with zero invoices")


> Empty result is a valid answer — document it. The pattern (LEFT JOIN + `IS NULL`) is how you find orphans for any schema.

Equivalent NOT IN (same intent, set-membership machinery — Solution 2 asserts both counts agree):


In [ ]:
no_buy2 = pd.read_sql("""
SELECT * FROM Customer
WHERE CustomerId NOT IN (SELECT DISTINCT CustomerId FROM Invoice)
""", conn)
assert len(no_buy) == len(no_buy2)


> **Pitfall:** SQL NULL means *unknown*, not zero. Two traps hide here: (1) if the `NOT IN` subquery ever contains a single NULL, the predicate is never true and the query returns **zero rows with no error** — prefer `NOT EXISTS` or `LEFT JOIN … IS NULL`; (2) `SUM`/`AVG` ignore NULLs, so a LEFT JOIN non-match aggregates to NULL, not 0 — wrap in `COALESCE` (as the spend variant does) before charting, or your bar dips to “missing” instead of zero.

**Variant for a non-empty demo** — customers whose total spend is below the median:


In [ ]:
cust_spend = pd.read_sql("""
SELECT c.CustomerId,
       c.FirstName || ' ' || c.LastName AS name,
       COALESCE(ROUND(SUM(i.Total), 2), 0) AS spend
FROM Customer c
LEFT JOIN Invoice i ON i.CustomerId = c.CustomerId
GROUP BY c.CustomerId
ORDER BY spend
""", conn)
print(cust_spend.head(5).to_string(index=False))
print("median spend:", cust_spend["spend"].median())


**What to notice:**
- **0 customers with zero invoices** — Chinook has no orphans; an empty DataFrame is the correct, reportable result.
- The two formulations must agree; `assert len(no_buy) == len(no_buy2)` fails loudly if they ever diverge.
- `COALESCE(ROUND(SUM(i.Total), 2), 0)` is what turns the LEFT JOIN’s NULL aggregate into a spend of 0 for an orphan — the median printed here spans all **59** customers, buyers and non-buyers alike.

---

### 4. Monthly revenue trend

SQLite has no bare DATE_TRUNC; `strftime('%Y-%m', InvoiceDate)` buckets the text timestamps into calendar months. The query returns a tidy frame — `ym`, `revenue`, `invoices` — small enough for matplotlib to plot directly as categories against dollars.


In [ ]:
monthly = pd.read_sql("""
SELECT strftime('%Y-%m', InvoiceDate) AS ym,
       ROUND(SUM(Total), 2) AS revenue,
       COUNT(*) AS invoices
FROM Invoice
GROUP BY ym
ORDER BY ym
""", conn)
print(monthly.head(6).to_string(index=False))
print("months:", len(monthly), "total:", monthly["revenue"].sum().round(2))


Plot the tidy monthly frame — rotate the x labels so all 60 months stay readable:


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(monthly["ym"], monthly["revenue"], marker="o", ms=3)
ax.set_title("Chinook monthly revenue")
ax.set_ylabel("Revenue ($)")
plt.xticks(rotation=90, fontsize=6)
fig.tight_layout(); fig.savefig("monthly_revenue.png", dpi=120)
from IPython.display import display
display(fig)
print("saved monthly_revenue.png")


**What to notice:**
- `months: 60`, `total: 2328.60` — the same invoices as Section 1, regrouped; totals must reconcile across groupings.
- Peaks: **2022-01 (52.62)**, **2023-04 (51.62)**, **2023-06 (50.62)**; a dip at **2023-11 (23.76)**.
- Mean month = 2328.60 / 60 = **38.81** (the Follow-up 1 check), so the high months sit well above average.

---

### 5. Bonus: top genres by revenue

Same aggregation, different join path: `InvoiceLine → Track → Genre`. Ranking genres by summed line revenue shows where money concentrates before anyone argues over a single “best” genre.


In [ ]:
genres = pd.read_sql("""
SELECT g.Name AS genre,
       ROUND(SUM(il.Quantity * il.UnitPrice), 2) AS revenue
FROM InvoiceLine il
JOIN Track t ON t.TrackId = il.TrackId
JOIN Genre g ON g.GenreId = t.GenreId
GROUP BY g.GenreId
ORDER BY revenue DESC
LIMIT 3
""", conn)
print(genres.to_string(index=False))


**What to notice:**
- **Rock 826.65 · Latin 382.14 · Metal 261.36** — Rock alone is 826.65 / 2328.60 ≈ 35% of store revenue (Follow-up 2 checks 0.355).
- Grouping by `g.GenreId` (not the name) keeps future duplicate genre names from collapsing — the same key-not-name habit as Section 2.
- `conn.close()` releases the file handle; SQLite otherwise holds its lock longer than needed.


In [ ]:
conn.close()


---

## Exercises

Do not modify the validation function. Write your code in the exercise cells, run those cells to verify, then run the validation cell. It checks the logic of your function against small cases. A different implementation is fine. It does not require the same lines of code or an exact number. If a check fails, the lab page says what is wrong and what to change. It does not give the answer.

### Exercise 1 — Top 5 tracks by revenue
Run the Section 2 query. List track name and revenue for all 5.


**Follow-up:** Divide total invoice revenue by the number of months. Return that mean monthly revenue, not the revenue of one invoice.

<details>
<summary>Hint</summary>

Join `InvoiceLine → Track → Album → Artist`, `SUM(Quantity*UnitPrice)`, `ORDER BY revenue DESC LIMIT 5`.
</details>

### Exercise 2 — Customers with no purchases
LEFT JOIN customers to invoices; count rows where invoice is NULL.


**Follow-up:** Divide Rock revenue by total revenue. Return Rock's share of revenue, not the Rock revenue amount.

<details>
<summary>Hint</summary>

`WHERE i.InvoiceId IS NULL` after `LEFT JOIN Invoice`.
</details>

### Exercise 3 — Monthly revenue trend
Group invoices by `strftime('%Y-%m', InvoiceDate)`. How many months? What is total revenue?


**Follow-up:** Compare mean vs median customer spend — which way is it skewed?

<details>
<summary>Hint</summary>

`SELECT strftime('%Y-%m', InvoiceDate) AS ym, ROUND(SUM(Total),2) … GROUP BY ym ORDER BY ym`.
</details>

---


In [ ]:
def avg_invoice():
    """Divide total invoice revenue by the number of months. Return that mean monthly revenue, not the revenue of one invoice."""
    pass


In [ ]:
def rock_revenue():
    """Add only the revenue from tracks in the Rock genre. Return that Rock revenue amount, not Rock's share of all revenue."""
    pass


In [ ]:
def rock_share():
    """Divide Rock revenue by total revenue. Return Rock's share of revenue, not the Rock revenue amount."""
    pass


In [ ]:
def mean_spend():
    """Divide total invoice revenue by the number of customers. Return that mean spend per customer, not the mean of single invoice lines."""
    pass


In [ ]:
kaa_check()  # prints your answers so you can compare them with the expected outputs


### What to learn next
- Window functions: `SUM(...) OVER (PARTITION BY …)` for running revenue.
- CTEs (`WITH`) for multi-step pivots.
- EXPLAIN QUERY PLAN on the invoice tables.
- Then SQLAlchemy / DuckDB for larger-than-memory analytics.
- Cheat sheet: connect → explore `sqlite_master` → JOIN path → GROUP BY → export.

*Files in this folder: `chinook.sqlite` · optional outputs `top_tracks.csv`, `monthly_revenue.png`.*


---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
